In [1]:
import json
from pyserini.search import FaissSearcher, LuceneSearcher, LuceneImpactSearcher
from pyserini.search.faiss import AutoQueryEncoder,AnceQueryEncoder,TctColBertQueryEncoder,AggretrieverQueryEncoder
from pyserini.search import get_topics, get_qrels
from tqdm import tqdm

import numpy as np
import random

topics = get_topics('dl19-passage')
qrels = get_qrels('dl19-passage')

with open('GOLFer_dl19/hypothesis_documents_dl19_5_qualified', 'r') as file:
    hypothesis_documents_dl19_qualified = json.load(file)

In [2]:
#1.0 bm25

searcher = LuceneSearcher.from_prebuilt_index("msmarco-v1-passage")

with open('GOLFer_dl19/dl19_bm25', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_bm25
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_bm25
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_bm25

Dec 15, 2024 5:08:17 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
100%|██████████| 43/43 [00:02<00:00, 15.32it/s]


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_bm25']
Results:
map                   	all	0.3013
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_bm25']
Results:
ndcg_cut_10           	all	0.5058
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000', '/root/.c

In [3]:
#1.1 bm25+GOLFer

searcher = LuceneSearcher.from_prebuilt_index("msmarco-v1-passage")

with open('GOLFer_dl19/dl19_bm25_mine', 'w')  as f:
    for i in range(len(hypothesis_documents_dl19_qualified)):
        qid=hypothesis_documents_dl19_qualified[i][0]
        question=hypothesis_documents_dl19_qualified[i][1]+'.'
        question=question*20
        for j in range(5):
            question=question+hypothesis_documents_dl19_qualified[i][j+2][0]
        hits = searcher.search(question, k=1000)
        rank = 0
        for hit in hits:
            rank += 1
            f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_bm25_mine
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_bm25_mine
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_bm25_mine

/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_bm25_mine']
Results:
map                   	all	0.3912
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_bm25_mine']
Results:
ndcg_cut_10           	all	0.5953
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000',

In [4]:
#2.0 ANCE
encoder = AnceQueryEncoder(encoder_dir='autodl-tmp/ance-msmarco-passage', pooling='mean')
searcher = FaissSearcher('autodl-tmp/msmarco-v1-passage.ance', encoder)

with open('GOLFer_dl19/dl19_ance', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_ance
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_ance
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_ance

/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/root/miniconda3/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
100%|██████████| 43/43 [01:41<00:00,  2.35s/it]


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_ance']
Results:
map                   	all	0.3710
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_ance']
Results:
ndcg_cut_10           	all	0.6452
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000', '/root/.c

In [5]:
#2.1 ANCE+GOLFer

encoder = AnceQueryEncoder(encoder_dir='autodl-tmp/ance-msmarco-passage', pooling='mean')
searcher = FaissSearcher('autodl-tmp/msmarco-v1-passage.ance', encoder)

def encode_weight(query, hypothesis_documents_withweight):
    coe=0.4/np.sum([[row[1]] for row in hypothesis_documents_withweight])
    prob=[[np.array(0.6)]]+[[row[1]*coe] for row in hypothesis_documents_withweight]
    hypothesis_documents=[row[0] for row in hypothesis_documents_withweight]
    all_emb_c = []
    for hypothesis_document in [query]+hypothesis_documents:
        c=hypothesis_document
        c_emb = encoder.encode(c)
        all_emb_c.append(np.array(c_emb))
    all_emb_c = np.array(all_emb_c)
    weighted_emb_c = np.sum(prob*all_emb_c, axis=0)
    hyde_vector = weighted_emb_c.reshape((1, len(weighted_emb_c)))
    return hyde_vector
    
with open('GOLFer_dl19/dl19_ance_mine', 'w')  as f:
    for i in range(len(hypothesis_documents_dl19_qualified)):
        qid=hypothesis_documents_dl19_qualified[i][0]
        encodedByWeight=encode_weight(hypothesis_documents_dl19_qualified[i][1],hypothesis_documents_dl19_qualified[i][2:])
        hits = searcher.search(encodedByWeight, k=1000)
        rank = 0
        for hit in hits:
            rank += 1
            f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_ance_mine
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_ance_mine
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_ance_mine

/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_ance_mine']
Results:
map                   	all	0.4412
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_ance_mine']
Results:
ndcg_cut_10           	all	0.7129
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000',

In [6]:
#3.0 tct_colbert

encoder = TctColBertQueryEncoder(encoder_dir='autodl-tmp/tct_colbert-v2-msmarco', pooling='mean') 
searcher = FaissSearcher('autodl-tmp/tct_colbert-v2_index', encoder)

with open('GOLFer_dl19/dl19_colbert', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_colbert
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_colbert
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_colbert

100%|██████████| 43/43 [01:47<00:00,  2.49s/it]


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_colbert']
Results:
map                   	all	0.4098
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_colbert']
Results:
ndcg_cut_10           	all	0.6843
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000', '/r

In [7]:
#3.1 tct_colbert+GOLFer

encoder = TctColBertQueryEncoder(encoder_dir='autodl-tmp/tct_colbert-v2-msmarco', pooling='mean') 
searcher = FaissSearcher('autodl-tmp/tct_colbert-v2_index', encoder)

def encode_weight(query, hypothesis_documents_withweight):
    coe=0.4/np.sum([[row[1]] for row in hypothesis_documents_withweight])
    prob=[[np.array(0.6)]]+[[row[1]*coe] for row in hypothesis_documents_withweight]
    hypothesis_documents=[row[0] for row in hypothesis_documents_withweight]
    all_emb_c = []
    for hypothesis_document in [query]+hypothesis_documents:
        c=hypothesis_document
        c_emb = encoder.encode(c)
        all_emb_c.append(np.array(c_emb))
    all_emb_c = np.array(all_emb_c)
    weighted_emb_c = np.sum(prob*all_emb_c, axis=0)
    hyde_vector = weighted_emb_c.reshape((1, len(weighted_emb_c)))
    return hyde_vector
    
with open('GOLFer_dl19/dl19_colbert_mine', 'w')  as f:
    for i in range(len(hypothesis_documents_dl19_qualified)):
        qid=hypothesis_documents_dl19_qualified[i][0]
        encodedByWeight=encode_weight(hypothesis_documents_dl19_qualified[i][1],hypothesis_documents_dl19_qualified[i][2:])
        hits = searcher.search(encodedByWeight, k=1000)
        rank = 0
        for hit in hits:
            rank += 1
            f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_colbert_mine
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_colbert_mine
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_colbert_mine


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_colbert_mine']
Results:
map                   	all	0.4824
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_colbert_mine']
Results:
ndcg_cut_10           	all	0.7334
/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.

In [8]:
#4.0 Aggretriever_distilbert

encoder = AggretrieverQueryEncoder(encoder_dir='autodl-tmp/aggretriever-distilbert', pooling='mean')
searcher = FaissSearcher('autodl-tmp/aggretriever-distillbert-index', encoder)

with open('GOLFer_dl19/dl19_distillbert', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_distillbert
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_distillbert
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_distillbert

100%|██████████| 43/43 [01:46<00:00,  2.47s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_distillbert']
Results:
map                   	all	0.4301


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_distillbert']
Results:
ndcg_cut_10           	all	0.6816


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_distillbert']
Results:
recall_1000           	all	0.8023


In [9]:
#4.1 Aggretriever_distilbert+GOLFer

encoder = AggretrieverQueryEncoder(encoder_dir='autodl-tmp/aggretriever-distilbert', pooling='mean')
searcher = FaissSearcher('autodl-tmp/aggretriever-distillbert-index', encoder)

def encode_weight(query, hypothesis_documents_withweight):
    coe=0.4/np.sum([[row[1]] for row in hypothesis_documents_withweight])
    prob=[[np.array(0.6)]]+[[row[1]*coe] for row in hypothesis_documents_withweight]
    hypothesis_documents=[row[0] for row in hypothesis_documents_withweight]
    all_emb_c = []
    for hypothesis_document in [query]+hypothesis_documents:
        c=hypothesis_document
        c_emb = encoder.encode(c)
        all_emb_c.append(np.array(c_emb))
    all_emb_c = np.array(all_emb_c)
    weighted_emb_c = np.sum(prob*all_emb_c, axis=0)
    hyde_vector = weighted_emb_c.reshape((1, len(weighted_emb_c)))
    return hyde_vector
    
with open('GOLFer_dl19/dl19_distillbert_mine', 'w')  as f:
    for i in range(len(hypothesis_documents_dl19_qualified)):
        qid=hypothesis_documents_dl19_qualified[i][0]
        encodedByWeight=encode_weight(hypothesis_documents_dl19_qualified[i][1],hypothesis_documents_dl19_qualified[i][2:])
        hits = searcher.search(encodedByWeight, k=1000)
        rank = 0
        for hit in hits:
            rank += 1
            f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_distillbert_mine
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_distillbert_mine
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_distillbert_mine


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_distillbert_mine']
Results:
map                   	all	0.4832


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_distillbert_mine']
Results:
ndcg_cut_10           	all	0.7028


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_distillbert_mine']
Results:
recall_1000           	all	0.8528


In [2]:
#5.0 aggretriever_cocondenser

encoder = AggretrieverQueryEncoder(encoder_dir='autodl-tmp/aggretriever-cocondenser', pooling='mean')
searcher = FaissSearcher('autodl-tmp/aggretriever_cocondenser_index', encoder)

with open('GOLFer_dl19/dl19_cocondenser', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_cocondenser
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_cocondenser
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_cocondenser

/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/root/miniconda3/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
100%|██████████| 43/43 [01:44<00:00,  2.44s/it]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable t

/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_cocondenser']
Results:
map                   	all	0.4350


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-m', 'ndcg_cut.10', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_cocondenser']
Results:
ndcg_cut_10           	all	0.6837


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/root/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'recall.1000', '/root/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'GOLFer_dl19/dl19_cocondenser']
Results:
recall_1000           	all	0.8078


In [ ]:
#5.1 aggretriever_cocondenser+GOLFer

encoder = AggretrieverQueryEncoder(encoder_dir='autodl-tmp/aggretriever-cocondenser', pooling='mean')
searcher = FaissSearcher('autodl-tmp/aggretriever_cocondenser_index', encoder)

def encode_weight(query, hypothesis_documents_withweight):
    coe=0.4/np.sum([[row[1]] for row in hypothesis_documents_withweight])
    prob=[[np.array(0.6)]]+[[row[1]*coe] for row in hypothesis_documents_withweight]
    hypothesis_documents=[row[0] for row in hypothesis_documents_withweight]
    all_emb_c = []
    for hypothesis_document in [query]+hypothesis_documents:
        c=hypothesis_document
        c_emb = encoder.encode(c)
        all_emb_c.append(np.array(c_emb))
    all_emb_c = np.array(all_emb_c)
    weighted_emb_c = np.sum(prob*all_emb_c, axis=0)
    hyde_vector = weighted_emb_c.reshape((1, len(weighted_emb_c)))
    return hyde_vector
    
with open('GOLFer_dl19/dl19_cocondenser_mine', 'w')  as f:
    for i in range(len(hypothesis_documents_dl19_qualified)):
        qid=hypothesis_documents_dl19_qualified[i][0]
        encodedByWeight=encode_weight(hypothesis_documents_dl19_qualified[i][1],hypothesis_documents_dl19_qualified[i][2:])
        hits = searcher.search(encodedByWeight, k=1000)
        rank = 0
        for hit in hits:
            rank += 1
            f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage GOLFer_dl19/dl19_cocondenser_mine
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage GOLFer_dl19/dl19_cocondenser_mine
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage GOLFer_dl19/dl19_cocondenser_mine

/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/root/miniconda3/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
